# Single-task MT Results Notebook — Full Version

Αυτό το notebook είναι για όλα τα single-task environments:

```text
button-press-v3
push-v3
pick-place-v3
basketball-v3
```

Σε αντίθεση με την προηγούμενη έκδοση, αυτό:

```text
1. Ψάχνει recursive μέσα στο current folder
2. Επιτρέπει manual paths για folders που έχουν διαφορετικό όνομα
3. Διαβάζει aggregate CSVs
4. Διαβάζει checkpoint evaluation CSVs, αν υπάρχουν
5. Βγάζει thesis-ready tables/plots
```

Βάλ’ το μέσα στο:

```text
single-task-mts/
```

Αν κάποια αποτελέσματα είναι έξω από αυτόν τον φάκελο, συμπλήρωσε τα paths στο `MANUAL_RESULT_PATHS`.


## 0. Setup

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

ROOT = Path(".")
print("Working directory:", ROOT.resolve())


## 1. Manual paths

Αν το notebook δεν βρίσκει κάποιο environment, βάλε εδώ το ακριβές path του CSV.

Παραδείγματα Windows paths:

```python
Path(r"C:/Users/Michalis/Desktop/basketball_v3(1)/basketball_v3_ppo_split_runs/results/basketball-v3_aggregate_results.csv")
```


In [ ]:

# Add or edit paths here if needed.
# The notebook will also search automatically with rglob below.

MANUAL_RESULT_PATHS = {
    "button-press-v3": [
        # Path(r"C:/Users/Michalis/Desktop/button_press_v3/button-press_v3_ppo_split_runs/results/button-press-v3_aggregate_results.csv"),
        # Path(r"C:/Users/Michalis/Desktop/button_press_v3/button-press_v3_ppo_split_runs/results/checkpoint_eval/button_checkpoint_eval_summary.csv"),
    ],
    "push-v3": [
        # Path(r"C:/Users/Michalis/Desktop/push_v3/push_v3_ppo_split_runs/results/push-v3_aggregate_results.csv"),
    ],
    "pick-place-v3": [
        # Path(r"C:/Users/Michalis/Desktop/pick-place/pick-place_v3_ppo_split_runs/results/pick-place-v3_aggregate_results.csv"),
    ],
    "basketball-v3": [
        # Path(r"C:/Users/Michalis/Desktop/basketball_v3(1)/basketball_v3_ppo_split_runs/results/basketball-v3_aggregate_results.csv"),
        # Path(r"C:/Users/Michalis/Desktop/basketball_v3(1)/basketball_v3_ppo_split_runs/results/checkpoint_eval/basketball_checkpoint_eval_summary.csv"),
    ],
}


## 2. Discover CSV files automatically

In [ ]:

# Aggregate result files from single-task scripts
aggregate_files = list(ROOT.rglob("*aggregate_results*.csv"))

# Per-episode files
per_episode_files = list(ROOT.rglob("*per_episode_results*.csv"))

# Checkpoint evaluation summaries
checkpoint_summary_files = list(ROOT.rglob("*checkpoint_eval_summary*.csv"))

# Also catch basketball-specific checkpoint summary name
checkpoint_summary_files += list(ROOT.rglob("*checkpoint*summary*.csv"))

# Remove duplicates
aggregate_files = sorted(set(aggregate_files))
per_episode_files = sorted(set(per_episode_files))
checkpoint_summary_files = sorted(set(checkpoint_summary_files))

print("Aggregate CSVs found:")
for p in aggregate_files:
    print(" -", p)

print("\nCheckpoint summary CSVs found:")
for p in checkpoint_summary_files:
    print(" -", p)

print("\nPer-episode CSVs found:")
for p in per_episode_files[:30]:
    print(" -", p)

if len(per_episode_files) > 30:
    print(f"... and {len(per_episode_files)-30} more")


## 3. Load aggregate results

In [ ]:

aggregate_dfs = []

# automatic aggregate files
for p in aggregate_files:
    try:
        df = pd.read_csv(p)
        df["source_file"] = str(p)
        aggregate_dfs.append(df)
    except Exception as e:
        print("Could not read aggregate:", p, e)

# manual aggregate files
for env_name, paths in MANUAL_RESULT_PATHS.items():
    for p in paths:
        if not p.exists():
            print("Manual path does not exist:", p)
            continue
        if "checkpoint" in p.name.lower():
            continue
        try:
            df = pd.read_csv(p)
            df["source_file"] = str(p)
            if "env_name" not in df.columns:
                df["env_name"] = env_name
            aggregate_dfs.append(df)
        except Exception as e:
            print("Could not read manual aggregate:", p, e)

aggregate = pd.concat(aggregate_dfs, ignore_index=True) if aggregate_dfs else pd.DataFrame()

print("Aggregate rows:", len(aggregate))
display(aggregate.head())
print(aggregate.columns.tolist() if not aggregate.empty else [])


## 4. Normalize environment/config columns

In [ ]:

if not aggregate.empty:
    agg = aggregate.copy()

    # Infer env column
    if "env_name" not in agg.columns:
        if "task_name" in agg.columns:
            agg["env_name"] = agg["task_name"]
        else:
            # Fallback from source file
            def infer_env_from_path(path):
                s = str(path).lower()
                if "button" in s:
                    return "button-press-v3"
                if "basketball" in s:
                    return "basketball-v3"
                if "pick" in s:
                    return "pick-place-v3"
                if "push" in s:
                    return "push-v3"
                return "unknown"
            agg["env_name"] = agg["source_file"].apply(infer_env_from_path)

    # Infer config column
    if "config_name" not in agg.columns:
        if "config" in agg.columns:
            agg["config_name"] = agg["config"]
        else:
            agg["config_name"] = "unknown"

    # Make sure success columns exist
    for col in ["train_success_rate", "test_success_rate"]:
        if col not in agg.columns:
            agg[col] = np.nan

    display(agg[["env_name", "config_name", "train_success_rate", "test_success_rate", "source_file"]].head(30))
else:
    agg = pd.DataFrame()
    print("No aggregate results loaded.")


## 5. Main single-task summary table

In [ ]:

if not agg.empty:
    summary = (
        agg.groupby(["env_name", "config_name"])
        .agg(
            mean_train_success=("train_success_rate", "mean"),
            mean_test_success=("test_success_rate", "mean"),
            std_test_success=("test_success_rate", "std"),
            runs=("test_success_rate", "count"),
        )
        .reset_index()
        .sort_values(["env_name", "mean_test_success"], ascending=[True, False])
    )

    display(summary)

    best_per_env = summary.sort_values(
        ["env_name", "mean_test_success", "mean_train_success"],
        ascending=[True, False, False],
    ).groupby("env_name").head(1)

    print("Best config per environment:")
    display(best_per_env)
else:
    print("No aggregate data.")


## 6. Plot test success by config for each environment

In [ ]:

if not agg.empty and "summary" in globals():
    for env in sorted(summary["env_name"].dropna().unique()):
        d = summary[summary["env_name"] == env].copy()

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(d["config_name"], d["mean_test_success"])
        ax.set_title(f"{env}: Single-task Test Success by Config")
        ax.set_xlabel("PPO config")
        ax.set_ylabel("Mean test success")
        ax.set_ylim(0, 1.05)
        ax.grid(True, axis="y", alpha=0.3)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("No summary to plot.")


## 7. Train vs test generalization gap

In [ ]:

if not agg.empty:
    agg["success_gap"] = agg["train_success_rate"] - agg["test_success_rate"]

    gap_summary = (
        agg.groupby(["env_name", "config_name"])
        .agg(
            mean_train_success=("train_success_rate", "mean"),
            mean_test_success=("test_success_rate", "mean"),
            mean_gap=("success_gap", "mean"),
            runs=("success_gap", "count"),
        )
        .reset_index()
        .sort_values(["env_name", "mean_test_success"], ascending=[True, False])
    )

    display(gap_summary)
else:
    print("No aggregate data.")


## 8. Load checkpoint evaluation summaries

In [ ]:

checkpoint_dfs = []

# automatic checkpoint files
for p in checkpoint_summary_files:
    try:
        df = pd.read_csv(p)
        df["source_file"] = str(p)
        checkpoint_dfs.append(df)
    except Exception as e:
        print("Could not read checkpoint summary:", p, e)

# manual checkpoint files
for env_name, paths in MANUAL_RESULT_PATHS.items():
    for p in paths:
        if not p.exists():
            continue
        if "checkpoint" not in p.name.lower():
            continue
        try:
            df = pd.read_csv(p)
            df["source_file"] = str(p)
            if "env_name" not in df.columns:
                df["env_name"] = env_name
            checkpoint_dfs.append(df)
        except Exception as e:
            print("Could not read manual checkpoint summary:", p, e)

checkpoints = pd.concat(checkpoint_dfs, ignore_index=True) if checkpoint_dfs else pd.DataFrame()

print("Checkpoint rows:", len(checkpoints))
display(checkpoints.head())
print(checkpoints.columns.tolist() if not checkpoints.empty else [])


## 9. Checkpoint learning curves

In [ ]:

if not checkpoints.empty:
    ckpt = checkpoints.copy()

    if "env_name" not in ckpt.columns:
        def infer_env_from_path(path):
            s = str(path).lower()
            if "basketball" in s:
                return "basketball-v3"
            if "button" in s:
                return "button-press-v3"
            if "pick" in s:
                return "pick-place-v3"
            if "push" in s:
                return "push-v3"
            return "unknown"
        ckpt["env_name"] = ckpt["source_file"].apply(infer_env_from_path)

    if "config_name" not in ckpt.columns:
        ckpt["config_name"] = "unknown"

    if "checkpoint_step" in ckpt.columns:
        ckpt["checkpoint_step_num"] = pd.to_numeric(ckpt["checkpoint_step"], errors="coerce")
    else:
        ckpt["checkpoint_step_num"] = np.nan

    if "total_timesteps" in ckpt.columns:
        ckpt["checkpoint_step_num"] = ckpt["checkpoint_step_num"].fillna(ckpt["total_timesteps"])

    ckpt["step_millions"] = ckpt["checkpoint_step_num"] / 1_000_000

    display(ckpt.head())

    if "test_success_rate" in ckpt.columns:
        for (env, cfg), d_cfg in ckpt.groupby(["env_name", "config_name"]):
            d_cfg = d_cfg.sort_values("step_millions")
            fig, ax = plt.subplots(figsize=(10, 5))

            if "split_id" in d_cfg.columns:
                for split_id in sorted(d_cfg["split_id"].dropna().unique()):
                    d = d_cfg[d_cfg["split_id"] == split_id].sort_values("step_millions")
                    ax.plot(d["step_millions"], d["test_success_rate"], marker="o", label=f"split {split_id}")
            else:
                ax.plot(d_cfg["step_millions"], d_cfg["test_success_rate"], marker="o", label=cfg)

            ax.set_title(f"{env} — {cfg}: Checkpoint Test Success")
            ax.set_xlabel("Training timesteps (millions)")
            ax.set_ylabel("Test success rate")
            ax.set_ylim(0, 1.05)
            ax.grid(True, alpha=0.3)
            ax.legend()
            plt.tight_layout()
            plt.show()
else:
    print("No checkpoint evaluation data loaded.")


## 10. Save outputs

In [ ]:

out_dir = Path("notebook_outputs_single_task_full")
fig_dir = out_dir / "figures"
out_dir.mkdir(exist_ok=True)
fig_dir.mkdir(exist_ok=True)

if not agg.empty:
    agg.to_csv(out_dir / "single_task_aggregate_loaded.csv", index=False)

if "summary" in globals():
    summary.to_csv(out_dir / "single_task_config_summary.csv", index=False)

if "gap_summary" in globals():
    gap_summary.to_csv(out_dir / "single_task_generalization_gap.csv", index=False)

if not checkpoints.empty:
    checkpoints.to_csv(out_dir / "single_task_checkpoint_summary_loaded.csv", index=False)

# Save main figures again
if "summary" in globals():
    for env in sorted(summary["env_name"].dropna().unique()):
        d = summary[summary["env_name"] == env].copy()

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(d["config_name"], d["mean_test_success"])
        ax.set_title(f"{env}: Single-task Test Success by Config")
        ax.set_xlabel("PPO config")
        ax.set_ylabel("Mean test success")
        ax.set_ylim(0, 1.05)
        ax.grid(True, axis="y", alpha=0.3)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        fig.savefig(fig_dir / f"{env.replace('-', '_')}_test_success_by_config.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

print("Saved outputs to:", out_dir)
